# 11. Korstabeller och chi-två-test

Hittills har vi jämfört medelvärden och andelar mellan *två* grupper. Nu breddar vi frågan till **samband mellan två
kvalitativa variabler**, som kan ha flera kategorier var: hänger åldersgrupp ihop med vilken kanal man handlar i?
Hänger utbildning ihop med partival?

Verktygen är **korstabellen** för att se sambandet och **chi-två-testet (χ²)** för att avgöra om det är mer än slump.

**Efter den här notebooken kan du**
- göra och läsa en korstabell, med antal och med radprocent
- räkna ut förväntade frekvenser under antagandet att inget samband finns
- genomföra och tolka ett chi-två-test, för hand och med scipy
- se vilka celler som driver ett samband

**Förkunskaper:** [8. Hypotesprövning och p-värden](../08_hypotesprovning/08_hypotesprovning.ipynb)
och [2. Variabeltyper](../02_variabeltyper/02_variabeltyper.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

## 11.1 Ett litet exempel att räkna på för hand

200 personer har fått frågan om de föredrar kaffe eller te. Finns det ett samband mellan åldersgrupp och dryck?

In [ ]:
small = pd.DataFrame({"Kaffe": [30, 70], "Te": [50, 50]}, index=["Under 40", "40 och över"])
small

Antalen är svåra att jämföra eftersom grupperna är olika stora. **Radprocent** visar fördelningen *inom* varje rad:

In [ ]:
small.div(small.sum(axis=1), axis=0).round(2)

Bland de yngre föredrar 38 % kaffe, bland de äldre 58 %. Det ser ut som ett samband. Men är det mer än slump?

### Vad hade vi väntat oss utan samband?

$H_0$: variablerna är **oberoende** (inget samband). Totalt föredrar 100 av 200, alltså hälften, kaffe.
Om ålder inte spelar någon roll borde hälften föredra kaffe *i båda åldersgrupperna*:

- Under 40: 80 personer, hälften → 40 kaffe, 40 te
- 40 och över: 120 personer, hälften → 60 kaffe, 60 te

Det är de **förväntade frekvenserna**. Den allmänna formeln är

$$E = \frac{\text{radsumma} \times \text{kolumnsumma}}{\text{totalt}}$$

In [ ]:
row_sums = small.sum(axis=1).to_numpy()
col_sums = small.sum(axis=0).to_numpy()
total = small.to_numpy().sum()

expected_small = pd.DataFrame(np.outer(row_sums, col_sums) / total, index=small.index, columns=small.columns)
expected_small

(`np.outer` multiplicerar varje radsumma med varje kolumnsumma.)

### Hur långt från det förväntade hamnade vi?

Testvariabeln mäter avståndet mellan observerade (O) och förväntade (E) frekvenser, summerat över alla celler:

$$\chi^2 = \sum \frac{(O - E)^2}{E}$$

Kvadraten gör att avvikelser uppåt och nedåt inte tar ut varandra (jämför med variansen i notebook 3).
Divisionen med E gör att en avvikelse på 10 väger tyngre där man väntade sig 40 än där man väntade sig 400.

In [ ]:
contributions = (small - expected_small) ** 2 / expected_small
display(contributions.round(2))

chi2_small = contributions.to_numpy().sum()
print(f"χ² = {chi2_small:.2f}")

Om $H_0$ är sann blir χ² litet (nära det förväntade). Ett stort χ² talar mot $H_0$. Hur stort är stort?
Det avgörs av **chi-två-fördelningen**, vars form beror på antalet **frihetsgrader**:

$$\text{frihetsgrader} = (\text{antal rader} - 1) \times (\text{antal kolumner} - 1)$$

Här (2 − 1) × (2 − 1) = 1.

In [ ]:
dof_small = (2 - 1) * (2 - 1)
p_small = stats.chi2.sf(chi2_small, df=dof_small)        # arean till höger om vårt χ²
print(f"p-värde = {p_small:.4f}")

p < 0,05, så $H_0$ förkastas: det finns ett statistiskt säkerställt samband mellan åldersgrupp och dryck.

scipy gör alla stegen på en gång:

In [ ]:
res = stats.chi2_contingency(small, correction=False)
print(f"χ² = {res.statistic:.2f},  frihetsgrader = {res.dof},  p-värde = {res.pvalue:.4f}")

(`correction=False` stänger av en kontinuitetskorrektion som scipy annars lägger på för tabeller med 2 × 2 celler.
Utan den får vi exakt samma svar som när vi räknade för hand.)

🧪 **Labba:** Ändra talen i tabellen `small` högst upp i avsnittet och kör om cellerna.
1. Pröva `"Kaffe": [40, 60], "Te": [40, 60]`. Vad blir χ², och varför?
2. Pröva `"Kaffe": [3, 7], "Te": [5, 5]`: samma andelar som från början men en tiondel så många personer. Vad händer med p-värdet?

<details>
<summary>💡 <b>Svar</b></summary>

1. χ² = 0. De observerade frekvenserna är exakt lika med de förväntade: andelen kaffedrickare är densamma i båda grupperna.
2. p-värdet blir stort. Samma mönster i ett litet stickprov kan lika gärna vara slump. (Här är dessutom de förväntade
   frekvenserna så små att testet inte är pålitligt, se förutsättningarna nedan.)

</details>

## 11.2 En större enkät

300 kunder har svarat på vilken kanal de helst handlar i. Datan simuleras så att kanalvalet beror på åldersgruppen.

In [ ]:
num_responses = 300
age_group = rng.choice(["18–34", "35–54", "55+"], num_responses)
probabilities = {"18–34": [0.6, 0.3, 0.1], "35–54": [0.4, 0.4, 0.2], "55+": [0.25, 0.4, 0.35]}
channel = [rng.choice(["App", "Webb", "Butik"], p=probabilities[group]) for group in age_group]

survey = pd.DataFrame({"age_group": age_group, "channel": channel})
survey.head()

Rådatan har en rad per person. `pd.crosstab` räknar ihop den till en korstabell:

In [ ]:
observed = pd.crosstab(survey["age_group"], survey["channel"])
pd.crosstab(survey["age_group"], survey["channel"], margins=True, margins_name="Totalt")

In [ ]:
row_percent = pd.crosstab(survey["age_group"], survey["channel"], normalize="index")
row_percent.round(2)

In [ ]:
row_percent.plot(kind="bar", stacked=True, rot=0)
plt.xlabel("Åldersgrupp")
plt.ylabel("Andel")
plt.legend(title="Kanal", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

Om det inte fanns något samband skulle de tre staplarna vara likadant indelade. Det är de inte.

✏️ **Din tur:** Innan vi låter scipy räkna:
1. Hur många frihetsgrader har testet?
2. Räkna för hand (med Python som miniräknare) ut den förväntade frekvensen för cellen *18–34 / App*, med hjälp av
   radsumman, kolumnsumman och totalen i tabellen ovan.

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
print("Frihetsgrader:", (3 - 1) * (3 - 1))
row_sum = observed.loc["18–34"].sum()
col_sum = observed["App"].sum()
print(f"Förväntad frekvens 18–34 / App: {row_sum} × {col_sum} / {observed.to_numpy().sum()} = {row_sum * col_sum / observed.to_numpy().sum():.1f}")
```

</details>

In [ ]:
chi2, p_value, dof, expected = stats.chi2_contingency(observed)
expected = pd.DataFrame(expected, index=observed.index, columns=observed.columns)

print("Förväntade frekvenser om H0 är sann:")
display(expected.round(1))
print(f"χ² = {chi2:.2f},  frihetsgrader = {dof},  p-värde = {p_value:.2g}")

alpha = 0.05
if p_value < alpha:
    print("Slutsats: förkasta H0. Det finns ett statistiskt säkerställt samband mellan åldersgrupp och kanal.")
else:
    print("Slutsats: H0 kan inte förkastas. Vi saknar belägg för ett samband.")

## 11.3 Var finns sambandet?

Testet säger bara *att* det finns ett samband, inte hur det ser ut. För det går vi tillbaka till tabellen.
Skillnaden O − E visar i vilka celler det finns fler eller färre än väntat:

In [ ]:
(observed - expected).round(1)

✏️ **Din tur:** Räkna ut varje cells bidrag till χ², alltså `(observed - expected)**2 / expected`.
Vilka celler bidrar mest? Beskriv sambandet i ord.

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
cell_contributions = (observed - expected) ** 2 / expected
display(cell_contributions.round(2))
print(f"Summa: {cell_contributions.to_numpy().sum():.2f}  (= χ²)")
# Störst bidrag kommer från de yngsta och de äldsta: 18–34 väljer appen oftare och butiken mer sällan än väntat,
# 55+ gör tvärtom. Mellangruppen ligger nära det förväntade.
```

</details>

## 11.4 Vad händer när inget samband finns?

🔮 **Gissa först:** om alla åldersgrupper har exakt samma preferenser, blir χ² då noll?

In [ ]:
same_probs = [0.4, 0.35, 0.25]         # samma kanalfördelning i alla åldersgrupper: H0 är sann
num_responses = 300                    # ← ändra: pröva 30 och 3000

age_sim = rng.choice(["18–34", "35–54", "55+"], num_responses)
channel_sim = rng.choice(["App", "Webb", "Butik"], num_responses, p=same_probs)
table_sim = pd.crosstab(age_sim, channel_sim)

res_sim = stats.chi2_contingency(table_sim)
display(table_sim)
print(f"χ² = {res_sim.statistic:.2f},  p-värde = {res_sim.pvalue:.3f}")

🧪 **Labba:** Kör cellen 10–20 gånger. χ² blir aldrig exakt noll, eftersom slumpen alltid ger små avvikelser.
Hur ofta får du p < 0,05? (Du vet svaret från [notebook 10](../10_typ1_typ2_fel/10_typ1_typ2_fel.ipynb).)

## 11.5 Förutsättningar och begränsningar

- Observationerna ska vara **oberoende**: varje person räknas en gång.
- De **förväntade frekvenserna** ska vara tillräckligt stora. Tumregel: minst 5 i varje cell. Annars kan man slå ihop kategorier.
- Testet säger inget om **styrkan** i sambandet. Med stora stickprov blir även svaga samband signifikanta. Titta på radprocenten.
- Testet tar ingen hänsyn till **ordning** mellan kategorier (ordinalskala).
- Ett samband är inte ett **orsakssamband**. Mer om det i nästa notebook.

✏️ **Din tur:** En kommun har frågat 400 invånare om de är nöjda med kollektivtrafiken:

```python
transit = pd.DataFrame({"Nöjd": [90, 70, 40], "Varken eller": [30, 40, 30], "Missnöjd": [20, 30, 50]},
                       index=["Centrum", "Förort", "Landsbygd"])
```

1. Formulera $H_0$ och $H_1$.
2. Räkna ut radprocent. Vad ser du?
3. Gör ett chi-två-test. Är förutsättningen om förväntade frekvenser uppfylld?
4. Formulera slutsatsen i ord.

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
transit = pd.DataFrame({"Nöjd": [90, 70, 40], "Varken eller": [30, 40, 30], "Missnöjd": [20, 30, 50]},
                       index=["Centrum", "Förort", "Landsbygd"])
# H0: inget samband mellan bostadsområde och nöjdhet.  H1: det finns ett samband.
display(transit.div(transit.sum(axis=1), axis=0).round(2))

res_transit = stats.chi2_contingency(transit)
print(f"χ² = {res_transit.statistic:.2f},  frihetsgrader = {res_transit.dof},  p-värde = {res_transit.pvalue:.2g}")
print(f"Minsta förväntade frekvens: {res_transit.expected_freq.min():.1f}")
# H0 förkastas: nöjdheten hänger ihop med var man bor. I centrum är 64 % nöjda, på landsbygden 33 %.
```

</details>

## Sammanfattning

- En **korstabell** visar två kvalitativa variabler samtidigt. **Radprocent** gör grupperna jämförbara.
- **Förväntad frekvens** = radsumma × kolumnsumma / totalt: vad vi skulle se utan samband.
- **χ² = Σ (O − E)² / E** mäter hur långt datan ligger från "inget samband".
- Frihetsgrader = (rader − 1) × (kolumner − 1).
- Testet visar *att* ett samband finns. *Hur* det ser ut ser du i radprocenten och i cellernas bidrag.

### Kontrollfrågor
1. En tabell har 4 rader och 3 kolumner. Hur många frihetsgrader?
2. Varför kan χ² aldrig bli negativt?
3. Ett chi-två-test på 50 000 personer ger p = 0,001. Betyder det att sambandet är starkt?

<details>
<summary>💡 <b>Svar</b></summary>

1. (4 − 1) × (3 − 1) = 6.
2. Varje term är en kvadrat delad med ett positivt tal.
3. Nej. Med så många observationer blir även mycket svaga samband signifikanta. Titta på hur mycket andelarna faktiskt skiljer sig.

</details>

**Nästa notebook:** [12. Korrelation](../12_korrelation/12_korrelation.ipynb)